# Clase 3 — Criptografía Simétrica por Bloques

> **Curso:** Criptografía Aplicada  
> **Objetivo:** Comprender el diseño, funcionamiento interno e implementación práctica de los cifrados simétricos por bloques más relevantes: DES, 3DES y AES, junto con sus modos de operación y vulnerabilidades derivadas de un uso incorrecto.

---

## Tabla de Contenidos

1. [Introducción: ¿Qué es un cifrado por bloques?](#1)  
2. [Primitivas: Redes de Feistel y Redes SP](#2)  
3. [DES — Data Encryption Standard](#3)  
4. [3DES — Triple DES](#4)  
5. [AES — Advanced Encryption Standard (Rijndael)](#5)  
   5.1 [Estructura global de AES](#5.1)  
   5.2 [SubBytes y la S-Box](#5.2)  
   5.3 [ShiftRows](#5.3)  
   5.4 [MixColumns](#5.4)  
   5.5 [AddRoundKey y expansión de clave](#5.5)  
   5.6 [Implementación paso a paso en Python](#5.6)  
6. [Modos de Operación](#6)  
   6.1 [ECB — Electronic Codebook](#6.1)  
   6.2 [CBC — Cipher Block Chaining](#6.2)  
   6.3 [CTR — Counter Mode](#6.3)  
   6.4 [GCM — Galois/Counter Mode (AEAD)](#6.4)  
   6.5 [Comparativa de modos](#6.5)  
7. [Padding: PKCS#7 y otras variantes](#7)  
8. [Ataque Padding Oracle](#8)  
9. [Consideraciones de rendimiento y seguridad práctica](#9)  
10. [Ejercicios propuestos](#10)  
11. [Referencias](#11)  


---
## Importaciones globales

In [ ]:
# Instalación de dependencias (ejecutar si es necesario)
# !pip install pycryptodome matplotlib numpy

import os, struct, time, textwrap
from functools import reduce

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

# Cryptodome (pycryptodome) — implementaciones de referencia
from Crypto.Cipher import DES, DES3, AES
from Crypto.Util.Padding import pad, unpad
from Crypto.Random import get_random_bytes

print("Dependencias cargadas correctamente ✔")


---
<a id='1'></a>
## 1. Introducción: ¿Qué es un cifrado por bloques?

Un **cifrado por bloques** (*block cipher*) es una función determinista que opera sobre grupos de bits de longitud fija (el "bloque") usando una clave secreta.

```
E : {0,1}^k × {0,1}^n  →  {0,1}^n
D : {0,1}^k × {0,1}^n  →  {0,1}^n
```

Donde `k` es el tamaño de la clave y `n` el tamaño del bloque.

### Propiedades deseables (Shannon, 1949)
| Propiedad | Descripción |
|-----------|-------------|
| **Confusión** | Cada bit del texto cifrado debe depender de múltiples bits de la clave (relación compleja). |
| **Difusión** | Cambiar un bit del texto plano debe afectar ~50 % de los bits del texto cifrado. |
| **Completeness** | Cada bit de salida depende de todos los bits de entrada. |
| **Avalanche effect** | Un cambio de 1 bit en la entrada provoca ~n/2 cambios en la salida. |

### Comparación con cifrados de flujo
| Característica | Cifrado por bloques | Cifrado de flujo |
|----------------|--------------------|--------------------|
| Unidad de proceso | Bloque fijo (64/128 bits) | Bit o byte |
| Estado | Sin estado (modo depende) | Con estado |
| Paralelismo | Según modo | Limitado |
| Ejemplos | AES, DES, 3DES | ChaCha20, RC4 (inseguro) |


In [ ]:
# ── Demostración del efecto avalancha ────────────────────────────────────────
def hamming_distance(b1: bytes, b2: bytes) -> int:
    """Número de bits que difieren entre dos cadenas de bytes."""
    return sum(bin(a ^ b).count('1') for a, b in zip(b1, b2))

key = get_random_bytes(16)          # AES-128
iv  = get_random_bytes(16)

plaintext_base = b"Hola mundo!! 128"  # 16 bytes = 1 bloque AES

diffs = []
for bit_pos in range(128):           # Flip cada bit del bloque
    mutated = bytearray(plaintext_base)
    byte_idx, bit_idx = divmod(bit_pos, 8)
    mutated[byte_idx] ^= (1 << bit_idx)

    c_base    = AES.new(key, AES.MODE_ECB).encrypt(plaintext_base)
    c_mutated = AES.new(key, AES.MODE_ECB).encrypt(bytes(mutated))
    diffs.append(hamming_distance(c_base, c_mutated))

plt.figure(figsize=(12, 4))
plt.plot(diffs, color='steelblue', linewidth=0.8)
plt.axhline(64, color='red', linestyle='--', label='50 % del bloque (64 bits)')
plt.xlabel("Bit del texto plano modificado"); plt.ylabel("Bits diferentes en cifrado")
plt.title("Efecto avalancha en AES-128 ECB")
plt.legend(); plt.tight_layout(); plt.show()

print(f"Media de bits diferentes: {np.mean(diffs):.1f} / 128 ({np.mean(diffs)/128*100:.1f} %)")


---
<a id='2'></a>
## 2. Primitivas: Redes de Feistel y Redes SP

### 2.1 Red de Feistel

Inventada por Horst Feistel en IBM (~1973). Divide el bloque en dos mitades (L, R) y aplica rondas:

```
L_{i+1} = R_i
R_{i+1} = L_i ⊕ F(R_i, K_i)
```

**Ventaja crítica:** la función `F` no necesita ser invertible. El descifrado usa las mismas operaciones en orden inverso.

```
 Cifrado        Descifrado
  L  | R          L  | R
  |  |            |  |
  +--⊕←F(R,K)   +--⊕←F(R,K)
  |  |            |  |
  swap            swap
  ...             ...
```

DES, 3DES, Blowfish, Twofish usan esta estructura.

### 2.2 Red de Sustitución-Permutación (SPN)

Aplica capas alternadas de:
- **SubBytes** (S-Box): sustitución no-lineal.
- **Permutación/Mezcla**: difusión lineal.
- **AddRoundKey**: XOR con subclave.

AES (Rijndael) es el ejemplo más conocido. La no-linealidad viene completamente de la S-Box.


In [ ]:
# ── Simulación simplificada de una red de Feistel de juguete (2 rondas) ──────
def toy_f(half: int, key: int) -> int:
    """Función de ronda simplificada de 8 bits."""
    return ((half * 5 + key) ^ (half >> 2)) & 0xFF

def feistel_encrypt(plaintext: int, keys: list) -> int:
    """Cifrado Feistel de 16 bits, 4 rondas."""
    L = (plaintext >> 8) & 0xFF
    R = plaintext & 0xFF
    for k in keys:
        L, R = R, L ^ toy_f(R, k)
    return (L << 8) | R

def feistel_decrypt(ciphertext: int, keys: list) -> int:
    """Descifrado: misma función, claves en orden inverso."""
    return feistel_encrypt(ciphertext, keys[::-1])

keys = [0x3A, 0x7F, 0x12, 0xC8]
plaintext = 0xABCD

ct = feistel_encrypt(plaintext, keys)
pt = feistel_decrypt(ct, keys)

print(f"Texto plano  : 0x{plaintext:04X}")
print(f"Cifrado      : 0x{ct:04X}")
print(f"Descifrado   : 0x{pt:04X}")
print(f"Correcto     : {pt == plaintext} ✔")


---
<a id='3'></a>
## 3. DES — Data Encryption Standard

### Historia
- Publicado por el NIST (entonces NBS) en **1977** como FIPS PUB 46.
- Diseñado por IBM con participación de la NSA.
- Bloques de **64 bits**, clave de **56 bits efectivos** (64 bits con 8 de paridad).
- **16 rondas** de Feistel.

### Estructura
```
Texto plano (64 bits)
    ↓
 IP (Initial Permutation)
    ↓
[16 rondas Feistel]
   L_i+1 = R_i
   R_i+1 = L_i ⊕ F(R_i, K_i)
    ↓
 IP⁻¹ (Final Permutation)
    ↓
Texto cifrado (64 bits)
```

### Función F de DES
1. **Expansión E**: 32 → 48 bits (duplica bits en los extremos).
2. **XOR** con la subclave de 48 bits.
3. **S-Boxes** (8 cajas × 6 bits → 4 bits): fuente de no-linealidad.
4. **Permutación P**: 32 bits → 32 bits.

### Debilidades y ataques
| Ataque | Año | Complejidad |
|--------|-----|-------------|
| Criptoanálisis diferencial (Biham & Shamir) | 1990 | 2⁴⁷ textos elegidos |
| Criptoanálisis lineal (Matsui) | 1993 | 2⁴³ textos conocidos |
| Fuerza bruta (EFF DES Cracker) | 1998 | 22 horas |

> ⚠️ **DES está considerado roto.** No debe usarse en ningún sistema nuevo.


In [ ]:
# ── DES con pycryptodome ─────────────────────────────────────────────────────
des_key = get_random_bytes(8)   # 64 bits (56 efectivos)
mensaje  = b"SecretMsg"         # 9 bytes → necesita padding

# Cifrar
cipher_enc = DES.new(des_key, DES.MODE_ECB)
padded = pad(mensaje, DES.block_size)          # PKCS#7
ciphertext = cipher_enc.encrypt(padded)

# Descifrar
cipher_dec = DES.new(des_key, DES.MODE_ECB)
decrypted  = unpad(cipher_dec.decrypt(ciphertext), DES.block_size)

print(f"Mensaje original : {mensaje}")
print(f"Clave DES (hex)  : {des_key.hex()}")
print(f"Cifrado (hex)    : {ciphertext.hex()}")
print(f"Descifrado       : {decrypted}")
print(f"Integridad       : {decrypted == mensaje} ✔")


In [ ]:
# ── Demostración: espacio de claves DES es trivialmente enumerable hoy ────────
import hashlib, random

def simulate_des_keyspace_fraction():
    """
    Simula cuántas claves DES podría probar una GPU moderna en 1 segundo.
    NVIDIA RTX 4090 ~ 1.5×10^10 DES/s (estimado real de benchmarks).
    """
    total_keys = 2**56
    gpu_rate   = 1_500_000_000     # 1.5 G claves/s (conservador)
    seconds    = total_keys / gpu_rate
    hours      = seconds / 3600

    print(f"Espacio de claves DES : 2^56 = {total_keys:,} claves")
    print(f"GPU moderna (~1.5G/s) : {gpu_rate:,} claves/segundo")
    print(f"Tiempo esperado       : {seconds:,.0f} s  ≈  {hours:,.1f} horas")
    print(f"Tiempo promedio (50%) : {hours/2:,.1f} horas")

simulate_des_keyspace_fraction()


---
<a id='4'></a>
## 4. 3DES — Triple DES (TDEA)

Ante la debilidad de DES, se propuso aplicarlo tres veces:

```
C = E_{K3}( D_{K2}( E_{K1}(P) ) )
```

### Modos de clave
| Opción | Bits efectivos | Uso |
|--------|---------------|-----|
| Tres claves independientes (K1≠K2≠K3) | 112 bits* | Recomendado |
| Dos claves (K1=K3) | 80 bits* | Legado |
| Una clave (K1=K2=K3) | 56 bits | Equivalente a DES |

*La seguridad real está limitada por el ataque meet-in-the-middle.

### ¿Por qué EDE (Encrypt-Decrypt-Encrypt)?
Si K1=K2=K3, `E(D(E(P)))=E(P)`, manteniendo compatibilidad con DES.

### Estado actual
- NIST desautorizó 3DES para nuevas aplicaciones en 2017 (NIST SP 800-131A).
- Aún presente en sistemas bancarios (SWIFT, EMV) por compatibilidad.
- Sujeto al ataque **Sweet32** (colisión de bloques de 64 bits, 2016).

> ⚠️ Migrar a AES-128 o superior en cualquier sistema nuevo.


In [ ]:
# ── Triple DES con pycryptodome ──────────────────────────────────────────────
tdes_key = get_random_bytes(24)    # 192 bits → 3 claves independientes de 64 bits
mensaje   = b"Triple DES demo!"   # 16 bytes exactos (2 bloques DES)

cipher_enc = DES3.new(tdes_key, DES3.MODE_CBC, iv=get_random_bytes(8))
ciphertext = cipher_enc.encrypt(mensaje)

cipher_dec = DES3.new(tdes_key, DES3.MODE_CBC, iv=cipher_enc.iv)
decrypted  = cipher_dec.decrypt(ciphertext)

print(f"Mensaje   : {mensaje}")
print(f"Cifrado   : {ciphertext.hex()}")
print(f"Descifrado: {decrypted}")
print(f"Correcto  : {decrypted == mensaje} ✔")


In [ ]:
# ── Sweet32: Riesgo de colisión con bloques de 64 bits ────────────────────────
import math

def birthday_collision_prob(block_size_bits: int, num_blocks: int) -> float:
    """Probabilidad de colisión por paradoja del cumpleaños."""
    n = 2 ** block_size_bits
    # Aproximación: 1 - exp(-q^2 / 2n)
    return 1 - math.exp(-num_blocks**2 / (2 * n))

# Para 3DES (bloque 64 bits) vs AES (bloque 128 bits)
gb_data = 32  # 32 GB de datos cifrados bajo la misma clave
blocks_64  = (gb_data * 1024**3) // 8   # bloques de 64 bits
blocks_128 = (gb_data * 1024**3) // 16  # bloques de 128 bits

p_des = birthday_collision_prob(64,  blocks_64)
p_aes = birthday_collision_prob(128, blocks_128)

print(f"Datos cifrados       : {gb_data} GB")
print(f"Bloques DES/3DES (64b): {blocks_64:,}")
print(f"P(colisión 3DES)     : {p_des:.4f}  ({p_des*100:.2f} %)")
print(f"Bloques AES (128b)   : {blocks_128:,}")
print(f"P(colisión AES)      : {p_aes:.2e}  (prácticamente 0)")


---
<a id='5'></a>
## 5. AES — Advanced Encryption Standard (Rijndael)

El NIST convocó un concurso público (1997-2001) para reemplazar DES. El ganador fue **Rijndael**, diseñado por Joan Daemen y Vincent Rijmen (Bélgica).

### Parámetros estándar
| Variante | Tamaño de bloque | Tamaño de clave | Rondas |
|----------|-----------------|----------------|--------|
| AES-128 | 128 bits | 128 bits | 10 |
| AES-192 | 128 bits | 192 bits | 12 |
| AES-256 | 128 bits | 256 bits | 14 |

> Rijndael permite bloques variables (128, 192, 256 bits), pero el estándar AES fija el bloque en 128 bits.

### Base matemática: GF(2⁸)
AES opera en el campo de Galois $\mathbb{F}_{2^8} = \mathbb{GF}(256)$, donde los elementos son polinomios de grado ≤ 7 con coeficientes en {0,1} y la aritmética se hace módulo el polinomio irreducible:

$$m(x) = x^8 + x^4 + x^3 + x + 1$$

La multiplicación de bytes en AES es multiplicación de polinomios módulo m(x).


<a id='5.1'></a>
### 5.1 Estructura global de AES

El estado de AES es una **matriz 4×4 de bytes** (16 bytes = 128 bits):

```
Estado = [[s00, s01, s02, s03],
           [s10, s11, s12, s13],
           [s20, s21, s22, s23],
           [s30, s31, s32, s33]]
```

Cada ronda aplica (en orden):
1. **SubBytes** — sustitución no-lineal (S-Box)
2. **ShiftRows** — rotación de filas
3. **MixColumns** — mezcla de columnas (excepto última ronda)
4. **AddRoundKey** — XOR con subclave

La primera operación es un `AddRoundKey` inicial (ronda 0), y la última ronda omite `MixColumns`.


In [ ]:
# ── Visualización de la estructura de ronda AES ──────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
labels = ["Estado", "SubBytes", "ShiftRows", "MixColumns", "AddRoundKey"]
colors_list = [
    [[0.9]*3]*4,       # gris
    ["#f4a261"]*4,     # naranja
    ["#2a9d8f"]*4,     # verde
    ["#e76f51"]*4,     # rojo
    ["#264653"]*4,     # azul oscuro
]

for ax, label, cols in zip(axes, labels, colors_list):
    for row in range(4):
        for col in range(4):
            color = cols[row] if isinstance(cols[0], str) else cols[row][col]
            ax.add_patch(plt.Rectangle((col, 3-row), 1, 1,
                         fc=color, ec='white', lw=2))
    ax.set_xlim(0,4); ax.set_ylim(0,4)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(label, fontsize=10, pad=4)

plt.suptitle("Operaciones por ronda en AES", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()


<a id='5.2'></a>
### 5.2 SubBytes y la S-Box

`SubBytes` aplica una tabla de sustitución fija (S-Box) a cada byte del estado. Proporciona la **no-linealidad** del cifrado, imprescindible para resistir criptoanálisis lineal y diferencial.

#### Construcción de la S-Box AES:
1. Calcular el inverso multiplicativo en $\mathbb{GF}(2^8)$ (el byte `0x00` se mapea a sí mismo).
2. Aplicar una transformación afín: $s' = A \cdot s \oplus c$

Donde $A$ es una matriz binaria fija de 8×8 y $c = 0x63$.


In [ ]:
# ── S-Box completa de AES ────────────────────────────────────────────────────
AES_SBOX = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16,
]

# Visualizar la S-Box como mapa de calor
sbox_matrix = np.array(AES_SBOX).reshape(16, 16)

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(sbox_matrix, cmap='viridis', aspect='auto')
plt.colorbar(im, ax=ax, label='Valor de salida (0-255)')
ax.set_xlabel("Nibble bajo (columna)"); ax.set_ylabel("Nibble alto (fila)")
ax.set_title("S-Box de AES — Mapa de calor")
ax.set_xticks(range(16)); ax.set_yticks(range(16))
ax.set_xticklabels([f'{i:X}' for i in range(16)])
ax.set_yticklabels([f'{i:X}' for i in range(16)])
plt.tight_layout(); plt.show()

# Verificar: ningún valor se mapea a sí mismo (excepto 0x00 en este análisis parcial)
fixed_points = [i for i in range(256) if AES_SBOX[i] == i]
print(f"Puntos fijos de la S-Box AES: {fixed_points}")
print(f"(0x63 es el punto fijo de 0x00 vía la transformación afín)")


<a id='5.3'></a>
### 5.3 ShiftRows

`ShiftRows` rota cíclicamente hacia la izquierda cada fila del estado:

| Fila | Desplazamiento |
|------|---------------|
| 0 | 0 posiciones |
| 1 | 1 posición |
| 2 | 2 posiciones |
| 3 | 3 posiciones |

```
Antes:  [s00, s01, s02, s03]    Después: [s00, s01, s02, s03]
        [s10, s11, s12, s13]             [s11, s12, s13, s10]
        [s20, s21, s22, s23]             [s22, s23, s20, s21]
        [s30, s31, s32, s33]             [s33, s30, s31, s32]
```

Garantiza que los bytes de cada columna procedan de columnas distintas del bloque original, contribuyendo a la **difusión**.


In [ ]:
# ── ShiftRows implementación y visualización ─────────────────────────────────
def shift_rows(state: list) -> list:
    """state: lista de 4 filas, cada fila con 4 bytes."""
    return [row[i:] + row[:i] for i, row in enumerate(state)]

def inv_shift_rows(state: list) -> list:
    return [row[-i:] + row[:-i] if i else row for i, row in enumerate(state)]

# Estado de ejemplo (letras para visualizar)
state = [
    ['s00','s01','s02','s03'],
    ['s10','s11','s12','s13'],
    ['s20','s21','s22','s23'],
    ['s30','s31','s32','s33'],
]

shifted = shift_rows(state)

print("Estado ANTES de ShiftRows:")
for row in state:   print(" ".join(f"{e:>3}" for e in row))
print("\nEstado DESPUÉS de ShiftRows:")
for row in shifted: print(" ".join(f"{e:>3}" for e in row))

# Verificar inversión
assert inv_shift_rows(shifted) == state, "Error en inv_shift_rows"
print("\n✔ inv_shift_rows(shift_rows(state)) == state")


<a id='5.4'></a>
### 5.4 MixColumns

`MixColumns` trata cada columna del estado como un polinomio de grado 3 sobre $\mathbb{GF}(2^8)$ y la multiplica por la matriz fija:

$$
M = \begin{pmatrix} 02 & 03 & 01 & 01 \\ 01 & 02 & 03 & 01 \\ 01 & 01 & 02 & 03 \\ 03 & 01 & 01 & 02 \end{pmatrix}
$$

Los valores 01, 02, 03 representan constantes en $\mathbb{GF}(2^8)$. La multiplicación por 02 equivale a un desplazamiento de bits con XOR condicional (xtime).

Esta operación garantiza que un cambio en cualquier byte de una columna afecte **todos** los bytes de esa columna.


In [ ]:
# ── Aritmética en GF(2^8) y MixColumns ───────────────────────────────────────
def xtime(a: int) -> int:
    """Multiplicación por 2 en GF(2^8) con polinomio reductor 0x1B."""
    result = (a << 1) & 0xFF
    if a & 0x80:
        result ^= 0x1B
    return result

def gf_mul(a: int, b: int) -> int:
    """Multiplicación general en GF(2^8) por el método de multiplicación por la izquierda."""
    result = 0
    for _ in range(8):
        if b & 1:
            result ^= a
        a = xtime(a)
        b >>= 1
    return result

def mix_column(col: list) -> list:
    """Mezcla una columna de 4 bytes."""
    a = col[:]
    return [
        gf_mul(0x02, a[0]) ^ gf_mul(0x03, a[1]) ^ a[2]          ^ a[3],
        a[0]               ^ gf_mul(0x02, a[1]) ^ gf_mul(0x03, a[2]) ^ a[3],
        a[0]               ^ a[1]               ^ gf_mul(0x02, a[2]) ^ gf_mul(0x03, a[3]),
        gf_mul(0x03, a[0]) ^ a[1]               ^ a[2]               ^ gf_mul(0x02, a[3]),
    ]

def mix_columns(state_cols: list) -> list:
    """state_cols: lista de 4 columnas, cada columna con 4 bytes."""
    return [mix_column(col) for col in state_cols]

# Verificación con el vector del estándar FIPS-197 (Apéndice B)
col_test = [0xd4, 0xbf, 0x5d, 0x30]
col_expected = [0x04, 0x66, 0x81, 0xe5]
result = mix_column(col_test)
print("Entrada :", [f'0x{x:02x}' for x in col_test])
print("Esperado:", [f'0x{x:02x}' for x in col_expected])
print("Obtenido:", [f'0x{x:02x}' for x in result])
print(f"Correcto: {result == col_expected} ✔")

# Tabla de multiplicación en GF(2^8)
print("\nTabla xtime (×2 en GF(2^8)):")
values = [0x01, 0x57, 0x83, 0xff]
for v in values:
    print(f"  xtime(0x{v:02x}) = 0x{xtime(v):02x}")


<a id='5.5'></a>
### 5.5 AddRoundKey y Expansión de Clave (Key Schedule)

`AddRoundKey` hace XOR del estado con una subclave de 128 bits derivada de la clave maestra. La **expansión de clave** de AES-128 produce 11 subclaves de 128 bits (44 palabras de 32 bits).

#### Algoritmo de expansión (AES-128):
```python
for i in range(4, 44):
    temp = W[i-1]
    if i % 4 == 0:
        temp = SubWord(RotWord(temp)) XOR Rcon[i//4]
    W[i] = W[i-4] XOR temp
```

Donde:
- `RotWord`: rotación cíclica de los 4 bytes.
- `SubWord`: aplicar S-Box a cada byte.
- `Rcon`: constante de ronda en GF(2⁸), potencias de 2.


In [ ]:
# ── Expansión de clave AES-128 ────────────────────────────────────────────────
RCON = [0x01, 0x02, 0x04, 0x08, 0x10, 0x20, 0x40, 0x80, 0x1b, 0x36]

def sub_word(word: list) -> list:
    return [AES_SBOX[b] for b in word]

def rot_word(word: list) -> list:
    return word[1:] + word[:1]

def key_expansion_aes128(key: bytes) -> list:
    """Devuelve lista de 11 subclaves, cada una como 16 bytes."""
    assert len(key) == 16
    W = [[key[4*i+j] for j in range(4)] for i in range(4)]

    for i in range(4, 44):
        temp = W[i-1][:]
        if i % 4 == 0:
            temp = sub_word(rot_word(temp))
            temp[0] ^= RCON[i//4 - 1]
        W.append([W[i-4][j] ^ temp[j] for j in range(4)])

    # Agrupar en 11 subclaves de 4 palabras
    return [bytes(b for word in W[4*r:4*r+4] for b in word) for r in range(11)]

# Ejemplo
test_key = bytes(range(16))   # 0x00 0x01 ... 0x0f
subkeys = key_expansion_aes128(test_key)

print(f"Clave maestra   : {test_key.hex()}")
print(f"Número de subclaves: {len(subkeys)}")
for i, sk in enumerate(subkeys):
    print(f"  Subclave[{i:2d}]  : {sk.hex()}")


<a id='5.6'></a>
### 5.6 Implementación AES-128 completa en Python puro

La siguiente implementación didáctica de AES-128 cifra y descifra un bloque de 16 bytes usando exclusivamente Python estándar. No está optimizada para producción (use pycryptodome o cryptography para eso), pero ilustra cada paso del algoritmo.


In [ ]:
# ── AES-128 didáctico ────────────────────────────────────────────────────────
AES_SBOX_INV = [0] * 256
for i, v in enumerate(AES_SBOX):
    AES_SBOX_INV[v] = i

def bytes_to_state(b: bytes) -> list:
    """16 bytes → matriz 4×4 (columna mayor)."""
    return [[b[r + 4*c] for c in range(4)] for r in range(4)]

def state_to_bytes(s: list) -> bytes:
    return bytes(s[r][c] for c in range(4) for r in range(4))

def add_round_key(state: list, subkey: bytes) -> list:
    sk = bytes_to_state(subkey)
    return [[state[r][c] ^ sk[r][c] for c in range(4)] for r in range(4)]

def sub_bytes(state: list) -> list:
    return [[AES_SBOX[state[r][c]] for c in range(4)] for r in range(4)]

def inv_sub_bytes(state: list) -> list:
    return [[AES_SBOX_INV[state[r][c]] for c in range(4)] for r in range(4)]

def _shift_rows_state(state: list) -> list:
    return [state[r][r:] + state[r][:r] for r in range(4)]

def _inv_shift_rows_state(state: list) -> list:
    return [state[r][-r:] + state[r][:-r] if r else state[r][:] for r in range(4)]

def _mix_col_enc(col):
    return [
        gf_mul(2,col[0])^gf_mul(3,col[1])^col[2]^col[3],
        col[0]^gf_mul(2,col[1])^gf_mul(3,col[2])^col[3],
        col[0]^col[1]^gf_mul(2,col[2])^gf_mul(3,col[3]),
        gf_mul(3,col[0])^col[1]^col[2]^gf_mul(2,col[3]),
    ]

def _mix_col_dec(col):
    return [
        gf_mul(0x0e,col[0])^gf_mul(0x0b,col[1])^gf_mul(0x0d,col[2])^gf_mul(0x09,col[3]),
        gf_mul(0x09,col[0])^gf_mul(0x0e,col[1])^gf_mul(0x0b,col[2])^gf_mul(0x0d,col[3]),
        gf_mul(0x0d,col[0])^gf_mul(0x09,col[1])^gf_mul(0x0e,col[2])^gf_mul(0x0b,col[3]),
        gf_mul(0x0b,col[0])^gf_mul(0x0d,col[1])^gf_mul(0x09,col[2])^gf_mul(0x0e,col[3]),
    ]

def _get_col(state, c): return [state[r][c] for r in range(4)]
def _set_col(state, c, col):
    for r in range(4): state[r][c] = col[r]

def mix_columns_state(state: list, inverse=False) -> list:
    s = [row[:] for row in state]
    for c in range(4):
        col = _get_col(s, c)
        _set_col(s, c, (_mix_col_dec if inverse else _mix_col_enc)(col))
    return s

def aes128_encrypt_block(plaintext: bytes, key: bytes) -> bytes:
    assert len(plaintext) == 16 and len(key) == 16
    subkeys = key_expansion_aes128(key)
    state   = bytes_to_state(plaintext)
    state   = add_round_key(state, subkeys[0])

    for rnd in range(1, 10):
        state = sub_bytes(state)
        state = _shift_rows_state(state)
        state = mix_columns_state(state)
        state = add_round_key(state, subkeys[rnd])

    # Última ronda (sin MixColumns)
    state = sub_bytes(state)
    state = _shift_rows_state(state)
    state = add_round_key(state, subkeys[10])
    return state_to_bytes(state)

def aes128_decrypt_block(ciphertext: bytes, key: bytes) -> bytes:
    assert len(ciphertext) == 16 and len(key) == 16
    subkeys = key_expansion_aes128(key)
    state   = bytes_to_state(ciphertext)
    state   = add_round_key(state, subkeys[10])

    for rnd in range(9, 0, -1):
        state = _inv_shift_rows_state(state)
        state = inv_sub_bytes(state)
        state = add_round_key(state, subkeys[rnd])
        state = mix_columns_state(state, inverse=True)

    state = _inv_shift_rows_state(state)
    state = inv_sub_bytes(state)
    state = add_round_key(state, subkeys[0])
    return state_to_bytes(state)

# ── Prueba con vector oficial FIPS-197 Appendix B ────────────────────────────
fips_key       = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
fips_plain     = bytes.fromhex("00112233445566778899aabbccddeeff")
fips_expected  = bytes.fromhex("69c4e0d86a7b04300d8a8b41b570efde")

ct = aes128_encrypt_block(fips_plain, fips_key)
pt = aes128_decrypt_block(ct, fips_key)

print(f"Clave      : {fips_key.hex()}")
print(f"Plano      : {fips_plain.hex()}")
print(f"Cifrado    : {ct.hex()}")
print(f"Esperado   : {fips_expected.hex()}")
print(f"FIPS OK    : {ct == fips_expected} ✔")
print(f"Descifrado : {pt.hex()}")
print(f"Round-trip : {pt == fips_plain} ✔")


---
<a id='6'></a>
## 6. Modos de Operación

Un cifrado por bloques cifra bloques individuales de tamaño fijo. Para mensajes de longitud arbitraria, necesitamos un **modo de operación** que defina cómo encadenar o combinar los bloques.

Los modos se clasifican en:
- **Modos de cifrado** (ECB, CBC): producen texto cifrado en bloques.
- **Modos de flujo de contador** (CTR, OFB, CFB): convierten el cifrador en un generador de flujo.
- **Modos autenticados (AEAD)** (GCM, CCM, SIV): cifrado + autenticación de integridad.


<a id='6.1'></a>
### 6.1 ECB — Electronic Codebook

```
C_i = E_K(P_i)   ∀ i
P_i = D_K(C_i)   ∀ i
```

- **Sin IV, sin estado.**
- Cada bloque se cifra independientemente.
- **Problema crítico:** bloques de texto plano iguales producen bloques cifrados iguales → revela patrones.

> ⚠️ **Nunca usar ECB** para datos que contengan patrones (imágenes, texto estructurado, datos binarios).


In [ ]:
# ── Vulnerabilidad ECB: "Tux penguin" en imágenes ────────────────────────────
# Generamos un patrón visual simplificado para demostrar el problema
import struct

def create_pattern_image(width=64, height=64) -> bytes:
    """Crea una imagen con franjas horizontales (simula imagen estructurada)."""
    data = bytearray()
    for y in range(height):
        for x in range(width):
            # Franjas: mitad negra, mitad blanca
            val = 0 if y < height // 2 else 255
            data.extend([val, val, val])
    return bytes(data)

def encrypt_ecb_bytes(data: bytes, key: bytes) -> bytes:
    """Cifra datos byte a byte en modo ECB (sin padding para múltiplos de 16)."""
    padded = pad(data, 16)
    return AES.new(key, AES.MODE_ECB).encrypt(padded)

key = get_random_bytes(16)
original = create_pattern_image(64, 64)
encrypted = encrypt_ecb_bytes(original, key)[:len(original)]  # Truncar al original

# Reconstruir como imagen de "intensidad" para visualizar
orig_arr = np.frombuffer(original, dtype=np.uint8).reshape(64, 64, 3)[:,:,0]
enc_arr  = np.frombuffer(encrypted[:64*64*3], dtype=np.uint8).reshape(64, 64, 3)[:,:,0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(orig_arr, cmap='gray', vmin=0, vmax=255)
ax1.set_title("Original (con patrón)", fontsize=12); ax1.axis('off')
ax2.imshow(enc_arr, cmap='gray', vmin=0, vmax=255)
ax2.set_title("Cifrado ECB (¡patrón visible!)", fontsize=12); ax2.axis('off')
plt.suptitle("Problema de ECB: los patrones se preservan", fontsize=13)
plt.tight_layout(); plt.show()

print("En ECB, bloques idénticos de texto plano → bloques idénticos de texto cifrado.")
print("Las franjas de la imagen siguen siendo visibles después del cifrado.")


<a id='6.2'></a>
### 6.2 CBC — Cipher Block Chaining

```
C_i = E_K(P_i ⊕ C_{i-1})   C_0 = IV
P_i = D_K(C_i) ⊕ C_{i-1}   C_0 = IV
```

- Requiere un **IV aleatorio** (no secreto, pero único por mensaje).
- Cada bloque depende de todos los anteriores → elimina patrones.
- **Cifrado secuencial** (no paralelizable), **descifrado paralelizable**.
- Vulnerabilidades: ataques por padding oracle (si se usan), CBC bit-flipping.


In [ ]:
# ── CBC: Implementación y demostración de propagación de errores ──────────────
def cbc_encrypt(plaintext: bytes, key: bytes, iv: bytes) -> bytes:
    padded = pad(plaintext, 16)
    cipher = AES.new(key, AES.MODE_CBC, iv=iv)
    return cipher.encrypt(padded)

def cbc_decrypt(ciphertext: bytes, key: bytes, iv: bytes) -> bytes:
    cipher = AES.new(key, AES.MODE_CBC, iv=iv)
    return unpad(cipher.decrypt(ciphertext), 16)

key = get_random_bytes(16)
iv  = get_random_bytes(16)
msg = b"Bloque_1_16bytesBloque_2_16bytesBloque_3_16bytes"

ct = cbc_encrypt(msg, key, iv)

# Corromper byte 5 del primer bloque cifrado
ct_corrupted = bytearray(ct)
ct_corrupted[5] ^= 0xFF
ct_corrupted = bytes(ct_corrupted)

try:
    pt_corrupted = AES.new(key, AES.MODE_CBC, iv=iv).decrypt(ct_corrupted)
    print("CBC - Propagación de errores:")
    print(f"  Bloque 1 descifrado (con error): {pt_corrupted[:16].hex()}")
    print(f"  Bloque 1 original              : {msg[:16].hex()}")
    print(f"  Bloque 2 descifrado (con error): {pt_corrupted[16:32]}")
    print(f"  Bloque 2 original              : {msg[16:32]}")
    print(f"  Bloque 3 descifrado (sin error): {pt_corrupted[32:48]}")
    print(f"  Bloque 3 original              : {msg[32:48]}")
    print("\n→ Un error en C_i corrompe completamente P_i y modifica 1 byte predecible de P_{i+1}.")
    print("  A partir de P_{i+2} la cadena se recupera.")
except Exception as e:
    print(f"Error: {e}")


<a id='6.3'></a>
### 6.3 CTR — Counter Mode

```
keystream_i = E_K(nonce || counter_i)
C_i = P_i ⊕ keystream_i
P_i = C_i ⊕ keystream_i
```

- Convierte AES en un **cifrado de flujo**.
- Cifrado y descifrado son la misma operación.
- **Completamente paralelizable** (acceso aleatorio).
- El **nonce debe ser único por mensaje/clave**. Reutilizarlo es catastrófico (two-time pad).


In [ ]:
# ── CTR: Acceso aleatorio y falla por reutilización de nonce ─────────────────
def ctr_keystream(key: bytes, nonce: bytes, num_blocks: int) -> bytes:
    """Genera num_blocks*16 bytes de keystream."""
    stream = b""
    for i in range(num_blocks):
        counter_block = nonce + struct.pack('>Q', i)  # 8 nonce + 8 counter = 16 bytes
        stream += AES.new(key, AES.MODE_ECB).encrypt(counter_block)
    return stream

key   = get_random_bytes(16)
nonce = get_random_bytes(8)
msg   = b"Este es un mensaje largo para demostrar acceso aleatorio en CTR"

n_blocks = (len(msg) + 15) // 16
ks  = ctr_keystream(key, nonce, n_blocks)
ct  = bytes(p ^ k for p, k in zip(msg, ks))
pt  = bytes(c ^ k for c, k in zip(ct, ks))

print(f"Mensaje   : {msg}")
print(f"Cifrado   : {ct.hex()}")
print(f"Descifrado: {pt}")
print(f"Correcto  : {pt == msg} ✔")

# ── Acceso aleatorio: descifrar solo el bloque 2 ──────────────────────────────
block_idx = 2
ks_block2 = ctr_keystream(key, nonce, block_idx+1)[block_idx*16:(block_idx+1)*16]
ct_block2 = ct[block_idx*16:block_idx*16+16]
pt_block2 = bytes(c ^ k for c, k in zip(ct_block2, ks_block2))
print(f"\nBloque {block_idx} descifrado directamente: '{pt_block2.decode(errors='replace')}'")

# ── Ataque por reutilización de nonce ────────────────────────────────────────
msg1 = b"Mensaje secreto!"
msg2 = b"XXXXXXXXXXXXX!!!"  # atacante conoce esto
ct1  = bytes(p ^ k for p, k in zip(msg1, ks[:16]))
ct2  = bytes(p ^ k for p, k in zip(msg2, ks[:16]))  # mismo nonce → mismo keystream

# XOR de cifrados = XOR de planos
xor_ct = bytes(a ^ b for a, b in zip(ct1, ct2))
xor_pt = bytes(a ^ b for a, b in zip(msg1, msg2))
print(f"\nXOR(ct1,ct2) = {xor_ct.hex()}")
print(f"XOR(pt1,pt2) = {xor_pt.hex()}")
print(f"Iguales      : {xor_ct == xor_pt} ← ¡reutilización de nonce filtra XOR de planos!")


<a id='6.4'></a>
### 6.4 GCM — Galois/Counter Mode (AEAD)

GCM combina CTR para cifrado con **GHASH** para autenticación de integridad en una sola pasada. Es un modo **AEAD** (*Authenticated Encryption with Associated Data*).

```
(ciphertext, tag) = GCM_Encrypt(key, nonce, plaintext, aad)
plaintext         = GCM_Decrypt(key, nonce, ciphertext, aad, tag)
                    ↑ error si tag inválido
```

- **IV/nonce de 96 bits** recomendado.
- **Tag de 128 bits** (puede truncarse a 96 o 64 bits con pérdida de seguridad).
- AAD (*Additional Authenticated Data*): datos que se autentican pero **no** se cifran (cabeceras, metadatos).
- Si el nonce se repite con la misma clave, la seguridad se compromete completamente (revelación de la clave de autenticación H).

#### GHASH
La autenticación usa multiplicación en $\mathbb{GF}(2^{128})$ con el polinomio $x^{128}+x^7+x^2+x+1$.


In [ ]:
# ── GCM con pycryptodome ──────────────────────────────────────────────────────
def gcm_encrypt(key: bytes, plaintext: bytes, aad: bytes = b"") -> tuple:
    """Retorna (nonce, ciphertext, tag)."""
    cipher = AES.new(key, AES.MODE_GCM)
    cipher.update(aad)
    ciphertext, tag = cipher.encrypt_and_digest(plaintext)
    return cipher.nonce, ciphertext, tag

def gcm_decrypt(key: bytes, nonce: bytes, ciphertext: bytes,
                tag: bytes, aad: bytes = b"") -> bytes:
    """Retorna plaintext o lanza ValueError si el tag es inválido."""
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    cipher.update(aad)
    return cipher.decrypt_and_verify(ciphertext, tag)

key     = get_random_bytes(16)
msg     = b"Datos confidenciales muy importantes"
aad     = b"usuario=alice;sesion=42"  # No se cifra, sí se autentica

nonce, ct, tag = gcm_encrypt(key, msg, aad)
print(f"Nonce   : {nonce.hex()} ({len(nonce)} bytes)")
print(f"Cifrado : {ct.hex()}")
print(f"Tag     : {tag.hex()} ({len(tag)} bytes)")

# Descifrado correcto
pt = gcm_decrypt(key, nonce, ct, tag, aad)
print(f"\nDescifrado: {pt}")
print(f"Correcto  : {pt == msg} ✔")

# ── Modificación del AAD (ataque de integridad) ───────────────────────────────
aad_tampered = b"usuario=eve;sesion=42"
try:
    gcm_decrypt(key, nonce, ct, tag, aad_tampered)
    print("\n¡ERROR! Se aceptó AAD manipulado")
except ValueError as e:
    print(f"\n✔ GCM rechazó AAD manipulado: {e}")

# ── Modificación del ciphertext ────────────────────────────────────────────────
ct_tampered = bytearray(ct); ct_tampered[0] ^= 0x01; ct_tampered = bytes(ct_tampered)
try:
    gcm_decrypt(key, nonce, ct_tampered, tag, aad)
    print("¡ERROR! Se aceptó ciphertext manipulado")
except ValueError as e:
    print(f"✔ GCM rechazó ciphertext manipulado: {e}")


<a id='6.5'></a>
### 6.5 Comparativa de modos de operación

| Modo | Paralelismo cifr. | Paralelismo descifr. | Acceso aleatorio | Autenticación | IV requerido | Seguro |
|------|:------------------:|:--------------------:|:----------------:|:-------------:|:------------:|:------:|
| ECB  | ✓ | ✓ | ✓ | ✗ | No | ❌ |
| CBC  | ✗ | ✓ | ✗ | ✗ | Sí | ⚠️ |
| CTR  | ✓ | ✓ | ✓ | ✗ | Sí (nonce) | ✓ |
| GCM  | ✓ | ✓ | ✓ | ✓ | Sí (nonce) | ✓✓ |
| CCM  | ✗ | ✗ | ✗ | ✓ | Sí (nonce) | ✓ |

> **Recomendación actual (2024):** Usar **AES-GCM** para la mayoría de aplicaciones. Si GCM no está disponible, **AES-CTR + HMAC-SHA256** como construcción MAC-then-Encrypt o Encrypt-then-MAC correctamente implementada.


In [ ]:
# ── Benchmark de modos ────────────────────────────────────────────────────────
import time

DATA_SIZE = 10 * 1024 * 1024   # 10 MB
data = get_random_bytes(DATA_SIZE)
key  = get_random_bytes(16)

modes_info = {
    'ECB': (AES.MODE_ECB, {}),
    'CBC': (AES.MODE_CBC, {'iv': get_random_bytes(16)}),
    'CTR': (AES.MODE_CTR, {'nonce': get_random_bytes(8)}),
    'GCM': (AES.MODE_GCM, {'nonce': get_random_bytes(12)}),
}

results = {}
for name, (mode, kwargs) in modes_info.items():
    padded = pad(data, 16) if name in ('ECB', 'CBC') else data
    start  = time.perf_counter()
    for _ in range(3):
        c = AES.new(key, mode, **kwargs)
        if name == 'GCM':
            c.encrypt_and_digest(padded)
        else:
            c.encrypt(padded)
    elapsed = (time.perf_counter() - start) / 3
    throughput = DATA_SIZE / elapsed / 1e6
    results[name] = throughput
    print(f"{name:4s}: {throughput:7.1f} MB/s")

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(results.keys(), results.values(),
              color=['#e63946','#457b9d','#2a9d8f','#264653'])
ax.set_ylabel("Throughput (MB/s)"); ax.set_title("Rendimiento de modos AES (10 MB, Python puro)")
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout(); plt.show()


---
<a id='7'></a>
## 7. Padding: PKCS#7 y otras variantes

Los cifrados por bloques requieren que el mensaje tenga longitud múltiplo del tamaño de bloque. El **padding** agrega bytes para completar el último bloque.

### PKCS#7 (RFC 5652)
Si faltan `n` bytes para completar el bloque, se agregan `n` bytes con valor `n`.

```
Mensaje: AA BB CC DD EE
Bloque (8 bytes): AA BB CC DD EE [03 03 03]
                                  ^^^^^^^^^  3 bytes de padding con valor 3
```

Si el mensaje ya es múltiplo del bloque, se agrega un bloque entero de padding:
```
Mensaje: AA BB CC DD EE FF 00 11 (8 bytes, bloque de 8)
Padding: 08 08 08 08 08 08 08 08 (bloque completo de valor 8)
```

### Otras variantes
| Esquema | Descripción | Uso |
|---------|-------------|-----|
| Zero padding | Rellena con 0x00 | Poco fiable (ambiguo si el mensaje termina en 0) |
| ANSI X9.23 | Un byte aleatorio + resto 0x00 + byte de longitud | Legado |
| ISO/IEC 7816-4 | 0x80 seguido de 0x00 | Smartcards |
| PKCS#5 | PKCS#7 restringido a bloques de 8 bytes | DES/3DES |


In [ ]:
# ── PKCS#7 manual ─────────────────────────────────────────────────────────────
def pkcs7_pad(data: bytes, block_size: int = 16) -> bytes:
    n = block_size - (len(data) % block_size)
    return data + bytes([n] * n)

def pkcs7_unpad(data: bytes, block_size: int = 16) -> bytes:
    if not data:
        raise ValueError("Datos vacíos")
    n = data[-1]
    if n == 0 or n > block_size:
        raise ValueError(f"Padding inválido: último byte = {n}")
    if len(data) < n:
        raise ValueError(f"Datos más cortos que el padding declarado ({len(data)} < {n})")
    if data[-n:] != bytes([n] * n):
        raise ValueError("Bytes de padding inconsistentes")
    return data[:-n]

test_cases = [
    b"",
    b"A",
    b"Hello",
    b"HelloWorld12345!",      # exactamente 16 bytes
    b"HelloWorld123456",      # 16 bytes → bloque completo de padding
]

print(f"{'Mensaje':<22} {'Padded (hex)':<48} {'Long.'}")
print("-" * 80)
for msg in test_cases:
    padded = pkcs7_pad(msg)
    unpadded = pkcs7_unpad(padded)
    assert unpadded == msg
    print(f"{repr(msg):<22} {padded.hex():<48} {len(padded)}")

print("\n✔ Todos los casos superaron la verificación")


---
<a id='8'></a>
## 8. Ataque Padding Oracle

El **Padding Oracle Attack** (Vaudenay, 2002) es un ataque de criptoanálisis adaptativo de texto cifrado elegido (CCA2) que explota la divulgación de si el padding es válido o no.

### Requisitos
1. El atacante puede enviar cualquier texto cifrado al servidor.
2. El servidor decifra y revela si el padding es **válido** (sin importar si responde con error 500, redirige, tarda más, etc.).

### Idea central para descifrar C_i
Recordemos el descifrado CBC:
```
P_i = D_K(C_i) ⊕ C_{i-1}
```

Si el atacante construye C'_{i-1} tal que:
```
D_K(C_i) ⊕ C'_{i-1} = [0x01]  (padding válido de 1 byte)
```

Entonces:
```
D_K(C_i) = C'_{i-1} ⊕ 0x01
P_i       = D_K(C_i) ⊕ C_{i-1}
```

### Complejidad
- Máximo **256 consultas por byte** (promedio 128).
- Para un bloque de 16 bytes: ~2048 consultas.
- Completamente práctico en redes con latencia normal.

### Defensas
1. **MAC-then-Encrypt correctamente:** verificar MAC antes de descifrar.
2. **Usar AEAD (GCM, ChaCha20-Poly1305):** el oracle desaparece.
3. **Tiempo constante en verificación de padding** (no suficiente por sí solo).
4. Nunca exponer errores de padding al cliente.


In [ ]:
# ── Simulación completa del Padding Oracle Attack ────────────────────────────
# Simulamos un servidor "oracle" que devuelve True si el padding CBC es válido.

SECRET_KEY = get_random_bytes(16)

def server_encrypt(plaintext: bytes) -> tuple:
    """Servidor: cifra con AES-CBC y devuelve (iv, ciphertext)."""
    iv = get_random_bytes(16)
    ct = AES.new(SECRET_KEY, AES.MODE_CBC, iv=iv).encrypt(pad(plaintext, 16))
    return iv, ct

def padding_oracle(iv: bytes, ciphertext: bytes) -> bool:
    """
    Oráculo: devuelve True si el descifrado tiene padding PKCS#7 válido.
    En un ataque real, esto podría inferirse de mensajes de error del servidor.
    """
    try:
        pt = AES.new(SECRET_KEY, AES.MODE_CBC, iv=iv).decrypt(ciphertext)
        pkcs7_unpad(pt, 16)
        return True
    except ValueError:
        return False

def padding_oracle_attack(iv: bytes, ciphertext: bytes) -> bytes:
    """
    Descifra ciphertext usando el padding oracle.
    Funciona bloque a bloque, byte a byte.
    """
    blocks = [iv] + [ciphertext[i:i+16] for i in range(0, len(ciphertext), 16)]
    n_cipher_blocks = len(blocks) - 1   # sin IV
    recovered = b""

    for blk_idx in range(1, len(blocks)):
        cipher_block = blocks[blk_idx]
        prev_block   = blocks[blk_idx - 1]

        # intermediate = D_K(cipher_block), lo que queremos encontrar
        intermediate = [0] * 16

        for byte_pos in range(15, -1, -1):
            pad_val = 16 - byte_pos  # 1, 2, ..., 16

            # Construir el bloque modificado C' con los bytes ya conocidos
            c_prime = bytearray(16)
            for k in range(byte_pos + 1, 16):
                c_prime[k] = intermediate[k] ^ pad_val

            # Probar los 256 valores posibles del byte byte_pos
            found = False
            for guess in range(256):
                c_prime[byte_pos] = guess
                if padding_oracle(bytes(c_prime), cipher_block):
                    # Verificar que no es un falso positivo del penúltimo byte
                    if byte_pos == 15:
                        # Confirmar modificando el byte anterior
                        c_verify = bytearray(c_prime)
                        c_verify[byte_pos - 1] ^= 0x01
                        if not padding_oracle(bytes(c_verify), cipher_block):
                            continue
                    intermediate[byte_pos] = guess ^ pad_val
                    found = True
                    break

            if not found:
                raise RuntimeError(f"Oracle no encontró valor en bloque {blk_idx}, byte {byte_pos}")

        # Recuperar el bloque de texto plano
        pt_block = bytes(intermediate[i] ^ prev_block[i] for i in range(16))
        recovered += pt_block

    return unpad(recovered, 16)

# ── Prueba del ataque ─────────────────────────────────────────────────────────
secret_msg = b"Mensaje secreto!"   # Exactamente 16 bytes
iv, ct     = server_encrypt(secret_msg)

print(f"Mensaje original  : {secret_msg}")
print(f"IV                : {iv.hex()}")
print(f"Ciphertext        : {ct.hex()}")
print(f"\nEjecutando Padding Oracle Attack...")

recovered = padding_oracle_attack(iv, ct)
print(f"\nMensaje recuperado: {recovered}")
print(f"Ataque exitoso    : {recovered == secret_msg} ✔")


In [ ]:
# ── Conteo de consultas al oracle ─────────────────────────────────────────────
def padding_oracle_attack_counted(iv, ciphertext):
    """Misma función pero cuenta las consultas al oráculo."""
    blocks   = [iv] + [ciphertext[i:i+16] for i in range(0, len(ciphertext), 16)]
    recovered = b""
    total_queries = 0

    for blk_idx in range(1, len(blocks)):
        cipher_block = blocks[blk_idx]
        prev_block   = blocks[blk_idx - 1]
        intermediate = [0] * 16

        for byte_pos in range(15, -1, -1):
            pad_val = 16 - byte_pos
            c_prime = bytearray(16)
            for k in range(byte_pos + 1, 16):
                c_prime[k] = intermediate[k] ^ pad_val

            for guess in range(256):
                c_prime[byte_pos] = guess
                total_queries += 1
                if padding_oracle(bytes(c_prime), cipher_block):
                    if byte_pos == 15:
                        c_verify = bytearray(c_prime); c_verify[byte_pos-1] ^= 1
                        total_queries += 1
                        if not padding_oracle(bytes(c_verify), cipher_block):
                            continue
                    intermediate[byte_pos] = guess ^ pad_val
                    break

        recovered += bytes(intermediate[i] ^ prev_block[i] for i in range(16))

    return unpad(recovered, 16), total_queries

recovered, queries = padding_oracle_attack_counted(iv, ct)
print(f"Consultas al oráculo : {queries}")
print(f"Teórico máximo       : {16 * 256}")
print(f"Teórico promedio     : {16 * 128}")
print(f"Bytes recuperados    : {len(recovered)}")
print(f"Consultas por byte   : {queries / len(recovered):.1f}")

# Visualizar distribución esperada de consultas
expected = [128] * 16
plt.figure(figsize=(8, 3))
plt.bar(range(16), expected, alpha=0.4, label='Media esperada (128)', color='gray')
plt.xlabel("Posición del byte en el bloque"); plt.ylabel("Consultas promedio")
plt.title("Consultas al oráculo por byte (estimado)")
plt.legend(); plt.tight_layout(); plt.show()


---
<a id='9'></a>
## 9. Consideraciones de rendimiento y seguridad práctica

### 9.1 Tamaños de clave recomendados (2024)

| Algoritmo | Seguridad (bits) | Estado |
|-----------|:----------------:|--------|
| DES | ~56 | ❌ Roto |
| 3DES-112 | ~80 | ⚠️ Legado |
| 3DES-168 | ~112 | ⚠️ Descontinuado |
| AES-128 | 128 | ✓ Seguro |
| AES-192 | 192 | ✓ Seguro |
| AES-256 | 256 | ✓✓ Resistente a computación cuántica (Grover) |

> Nota: Grover's algorithm reduce la seguridad efectiva de cifrados simétricos a la mitad para un adversario cuántico. AES-256 ofrece 128 bits de seguridad post-cuántica.

### 9.2 Reglas de oro para el uso correcto

1. **Usar siempre AEAD** (preferiblemente AES-GCM o ChaCha20-Poly1305).
2. **Nunca reutilizar el nonce/IV** con la misma clave.
3. **Generar IVs aleatoriamente** con un CSPRNG (`os.urandom`, `get_random_bytes`).
4. **No truncar el tag** de GCM por debajo de 96 bits.
5. **Rotar claves periódicamente** y limitar la cantidad de datos cifrados por clave.
6. **Usar bibliotecas auditadas** (cryptography, pycryptodome), no implementaciones propias.


In [ ]:
# ── Comparativa de seguridad: DES vs 3DES vs AES ────────────────────────────
algoritmos = {
    'DES'     : {'key_bits': 56,  'block_bits': 64,  'rounds': 16, 'seguro': False},
    '3DES-112': {'key_bits': 112, 'block_bits': 64,  'rounds': 48, 'seguro': False},
    '3DES-168': {'key_bits': 168, 'block_bits': 64,  'rounds': 48, 'seguro': False},
    'AES-128' : {'key_bits': 128, 'block_bits': 128, 'rounds': 10, 'seguro': True},
    'AES-192' : {'key_bits': 192, 'block_bits': 128, 'rounds': 12, 'seguro': True},
    'AES-256' : {'key_bits': 256, 'block_bits': 128, 'rounds': 14, 'seguro': True},
}

print(f"{'Algoritmo':<12} {'Clave (b)':<12} {'Bloque (b)':<12} {'Rondas':<8} {'Seguro'}")
print("-" * 56)
for name, info in algoritmos.items():
    seg = "✓" if info['seguro'] else "❌"
    print(f"{name:<12} {info['key_bits']:<12} {info['block_bits']:<12} {info['rounds']:<8} {seg}")

# ── Rendimiento relativo ──────────────────────────────────────────────────────
DATA = get_random_bytes(1 * 1024 * 1024)  # 1 MB

benchmarks = {}
# DES
key8 = get_random_bytes(8)
t = time.perf_counter()
DES.new(key8, DES.MODE_ECB).encrypt(pad(DATA, 8))
benchmarks['DES'] = 1 / (time.perf_counter() - t)

# 3DES
key24 = get_random_bytes(24)
t = time.perf_counter()
DES3.new(key24, DES3.MODE_ECB).encrypt(pad(DATA, 8))
benchmarks['3DES'] = 1 / (time.perf_counter() - t)

# AES-128, AES-256
for bits, klen in [('AES-128', 16), ('AES-256', 32)]:
    k = get_random_bytes(klen)
    t = time.perf_counter()
    AES.new(k, AES.MODE_ECB).encrypt(pad(DATA, 16))
    benchmarks[bits] = 1 / (time.perf_counter() - t)

# Normalizar a AES-128
base = benchmarks['AES-128']
print("\nRendimiento relativo (AES-128 = 1.0x):")
for alg, thr in benchmarks.items():
    print(f"  {alg:<10}: {thr/base:.2f}x ({thr:.1f} MB/s aprox.)")


In [ ]:
# ── Diagrama de decisión: ¿qué modo usar? ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')

steps = [
    (0.5, 0.9,  "¿Necesita autenticación?", "diamond"),
    (0.2, 0.7,  "NO: ¿Necesita acceso aleatorio?", "box"),
    (0.8, 0.7,  "SÍ → Usar AEAD", "box"),
    (0.1, 0.5,  "NO → CBC (con MAC externo)", "box"),
    (0.35, 0.5, "SÍ → CTR (con MAC externo)", "box"),
    (0.65, 0.7, "AES-GCM (preferido)", "box"),
    (0.93, 0.7, "ChaCha20-Poly1305", "box"),
]

facecolors = {
    "diamond": "#fff176",
    "box":     "#b3e5fc",
}

for x, y, label, shape in steps:
    bbox = dict(boxstyle="round,pad=0.4" if shape=="box" else "round4,pad=0.4",
                fc=facecolors[shape], ec="gray", lw=1.5)
    ax.text(x, y, label, ha='center', va='center', fontsize=9,
            bbox=bbox, transform=ax.transAxes, wrap=True)

# Flechas
arrow_props = dict(arrowstyle='->', color='#37474f', lw=1.5)
connections = [
    (0.5, 0.87, 0.25, 0.73), (0.5, 0.87, 0.78, 0.73),
    (0.2, 0.67, 0.1,  0.53), (0.2, 0.67, 0.35, 0.53),
    (0.78, 0.67, 0.65, 0.70), (0.78, 0.67, 0.93, 0.70),
]
for x1,y1,x2,y2 in connections:
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=arrow_props)

ax.set_title("Árbol de decisión: selección de modo de cifrado", fontsize=12, pad=10)
plt.tight_layout(); plt.show()


---
<a id='10'></a>
## 10. Ejercicios Propuestos

### Nivel básico
1. **Efecto avalancha de DES:** Repite el análisis del efecto avalancha (sección 1) usando DES en lugar de AES. ¿Cuántos bits cambian en promedio? ¿Hay diferencia estadística con AES?

2. **Extensión a AES-192/256:** Modifica `key_expansion_aes128` para soportar AES-192 (12 rondas, 6 palabras por ronda) y AES-256 (14 rondas). Valida con los vectores del FIPS-197 Apéndice C.

3. **Padding manual:** Implementa las funciones `ansi_x923_pad`/`unpad` y `iso7816_pad`/`unpad` y verifica que son inversas.

### Nivel intermedio
4. **CBC bit-flipping attack:** En el cifrado CBC de un mensaje como `"admin=false;user=alice"`, ¿cómo modificarías el ciphertext para que al descifrar el servidor lea `"admin=true ;user=alice"`? Implementa y demuestra el ataque.

5. **CTR nonce reuse:** Dado `C1 = P1 ⊕ KS` y `C2 = P2 ⊕ KS` (misma clave y nonce), si conoces `P2`, recupera `P1` completamente. Implementa la recuperación.

6. **GCM forgery con nonce repetido:** Si el mismo nonce se usa dos veces en GCM, el atacante puede recuperar la clave de autenticación H. Investiga el ataque y explica matemáticamente por qué ocurre.

### Nivel avanzado
7. **Ataque meet-in-the-middle a doble DES (2DES):** Implementa el ataque que muestra que 2DES solo ofrece ~57 bits de seguridad en lugar de 112. (Requiere ~2⁵⁶ de tiempo y espacio.)

8. **Padding Oracle en la práctica:** Configura un servidor Flask sencillo que tenga una vulnerabilidad de padding oracle y aplica el ataque de la sección 8 de forma remota.

9. **AES-GCM desde cero:** Implementa GHASH usando aritmética en GF(2¹²⁸) y construye el modo GCM completo sin usar pycryptodome. Valida con los vectores de NIST SP 800-38D.


In [ ]:
# ── Plantilla para el ejercicio 4: CBC bit-flipping ──────────────────────────
# Pista: En CBC, P_i = D_K(C_i) ⊕ C_{i-1}
# Si modificas C_{i-1}[j] ^= delta, entonces P_i[j] ^= delta (controlado)

def cbc_bitflip_demo():
    key = get_random_bytes(16)
    iv  = get_random_bytes(16)

    # Mensaje con estructura fija (bloques de 16 bytes)
    bloque0 = b"prefix_padding!!"   # 16 bytes (bloque 0 - padding del atacante)
    bloque1 = b"admin=false;xyz!"   # 16 bytes (bloque 1 - lo que queremos modificar)
    plaintext = bloque0 + bloque1

    ct = AES.new(key, AES.MODE_CBC, iv=iv).encrypt(plaintext)
    ct_array = bytearray(ct)

    # Queremos: 'admin=false;xyz!' → 'admin=true ;xyz!'
    #                    ^^^^^               ^^^^
    # Posición de 'false' en bloque1 = índice 6..10
    # Debemos modificar el BLOQUE 0 del ciphertext en esas posiciones

    original  = b"false"
    target    = b"true "
    for i, (o, t) in enumerate(zip(original, target)):
        pos_in_block0 = 6 + i
        ct_array[pos_in_block0] ^= o ^ t    # delta = o XOR t

    # Descifrar (bloque 0 quedará basura, bloque 1 tendrá la modificación)
    pt_mod = AES.new(key, AES.MODE_CBC, iv=iv).decrypt(bytes(ct_array))

    print("Bloque 0 descifrado (basura):", pt_mod[:16])
    print("Bloque 1 descifrado (atacado):", pt_mod[16:32])
    print("¿admin=true?:", b"admin=true" in pt_mod[16:32])

cbc_bitflip_demo()


---
<a id='11'></a>
## 11. Referencias

### Estándares
1. NIST FIPS 197 (2001) — *Advanced Encryption Standard (AES)*. https://csrc.nist.gov/publications/fips/fips197/fips-197.pdf
2. NIST FIPS 46-3 (1999) — *Data Encryption Standard (DES)*. https://csrc.nist.gov/publications/detail/fips/46/3/archive/1999-10-25
3. NIST SP 800-38A (2001) — *Recommendation for Block Cipher Modes of Operation*. https://csrc.nist.gov/publications/detail/sp/800-38a/final
4. NIST SP 800-38D (2007) — *Recommendation for Block Cipher Modes of Operation: GCM and GMAC*. https://csrc.nist.gov/publications/detail/sp/800-38d/final
5. NIST SP 800-131A Rev. 2 (2019) — *Transitioning the Use of Cryptographic Algorithms and Key Lengths*.

### Libros de texto
6. Daemen, J., & Rijmen, V. (2002). *The Design of Rijndael: AES — The Advanced Encryption Standard*. Springer. ISBN 978-3-540-42580-9.
7. Stinson, D.R., & Paterson, M. (2018). *Cryptography: Theory and Practice* (4th ed.). CRC Press.
8. Ferguson, N., Schneier, B., & Kohno, T. (2010). *Cryptography Engineering*. Wiley. ISBN 978-0-470-47424-2.
9. Boneh, D., & Shoup, V. (2023). *A Graduate Course in Applied Cryptography* (v0.6). https://toc.cryptobook.us/
10. Menezes, A., van Oorschot, P., & Vanstone, S. (1996). *Handbook of Applied Cryptography*. CRC Press. (gratuito en https://cacr.uwaterloo.ca/hac/)

### Artículos originales y ataques
11. Vaudenay, S. (2002). *Security Flaws Induced by CBC Padding*. EUROCRYPT 2002. (Padding Oracle original)
12. Biham, E., & Shamir, A. (1991). *Differential Cryptanalysis of DES-like Cryptosystems*. Journal of Cryptology, 4(1), 3–72.
13. Matsui, M. (1993). *Linear Cryptanalysis Method for DES Cipher*. EUROCRYPT 1993.
14. Bhargavan, K., & Leurent, G. (2016). *On the Practical (In)Security of 64-bit Block Ciphers*. CCS 2016. (Sweet32)
15. EFF. (1998). *Cracking DES: Secrets of Encryption Research, Wiretap Politics and Chip Design*.

### Recursos en línea
16. AES Animation — https://www.cryptoclub.org/tools/aesvisualizer.php
17. Dan Boneh's Cryptography I (Coursera) — https://www.coursera.org/learn/crypto
18. Cryptopals Challenges — https://cryptopals.com (ejercicios prácticos, sets 1–3 son especialmente relevantes para esta clase)
19. NIST Cryptographic Standards & Guidelines — https://csrc.nist.gov/projects/cryptographic-standards-and-guidelines
20. pycryptodome Documentation — https://pycryptodome.readthedocs.io/

---
> *Clase preparada para el Curso de Criptografía Aplicada. Para reportar errores o sugerencias, abrir un issue en el repositorio.*
